# Algorithms for massive datasets project

## 1) Data loading

In [ ]:
import os
import zipfile
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows
with zipfile.ZipFile("imdb-dataset-of-top-1000-movies-and-tv-shows.zip", "r") as zip_ref:
    zip_ref.extractall("imdb_data")

In [ ]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark import StorageLevel
from pyspark.sql.types import StructType, StructField, StringType,  DoubleType, IntegerType
import math
from collections import Counter
from itertools import combinations
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
spark = (
    SparkSession.builder.appName("IMDB MBA")
    .config("spark.python.worker.reuse", "true")
    .config("spark.python.worker.faulthandler.enabled", "true")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)
sc = spark.sparkContext

In [ ]:
imdb_schema = StructType([
    StructField("Poster_Link", StringType(), True),
    StructField("Series_Title", StringType(), True),
    StructField("Released_Year", StringType(), True),
    StructField("Certificate", StringType(), True),
    StructField("Runtime", StringType(), True),
    StructField("Genre", StringType(), True),
    StructField("IMDB_Rating", DoubleType(), True),
    StructField("Overview", StringType(), True),
    StructField("Meta_score", DoubleType(), True),
    StructField("Director", StringType(), True),
    StructField("Star1", StringType(), True),
    StructField("Star2", StringType(), True),
    StructField("Star3", StringType(), True),
    StructField("Star4", StringType(), True),
    StructField("No_of_Votes", IntegerType(), True),
    StructField("Gross", StringType(), True),
])
file_path = os.environ.get("IMDB_CSV_PATH","imdb_data/imdb_top_1000.csv")
df = spark.read.csv(file_path, header=True, inferSchema=True)
actors = ["Star1", "Star2", "Star3", "Star4"]
df = df.select(actors)
df.head(2)

## 2) Basket creation

In [ ]:
n_rows = df.count()
target_rows_partition = int(os.environ.get("TARGET_ROWS_PARTITION", "50000"))
num_partitions = max(
    sc.defaultParallelism * 2,
    math.ceil(n_rows / target_rows_partition)
)

baskets = (
    df.rdd
    .map(
          lambda row: tuple(
              sorted({
                  str(actor).strip()
                  for actor in row
                  if actor is not None
                  and str(actor).strip() != ""
              }) #set used to avoid actors repetitions
          )
      )
      .filter(
          lambda basket:
              len(basket) >= 2
      )
      .repartition(num_partitions)
      .persist(StorageLevel.MEMORY_AND_DISK
      )
)

n_baskets = baskets.count()

print("Number of baskets:", n_baskets)
print("Number of partitions:",baskets.getNumPartitions())

## 3) Preprocessing

In [ ]:
#setting min support
min_supp = 0.004
glob_supp = math.ceil(min_supp * n_baskets)

## 4) Apriori

In [ ]:
#done for candidate generation
def generate_candidates(previous_freq, k):  #k = itemset dimension

    previous_freq = sorted(previous_freq)
    previous_freq_set = set(previous_freq)

    candidates = set()

    for i in range(len(previous_freq)):

        for j in range(i + 1, len(previous_freq)):

            first = previous_freq[i]
            second = previous_freq[j] #comparison between itemsets pairs of dimension k-1

            if k > 2 and first[:-1] != second[:-1]:
                continue

            candidate = tuple( #used tuple to put in set
                sorted(
                set(first) | set(second)
                )
            )
            
            if len(candidate) != k:
                continue

            valid = True

            for subset in combinations(candidate, k - 1):

                if subset not in previous_freq_set:
                    valid = False
                    break

            if valid:
                candidates.add(candidate)

    return candidates


def local_apriori_pass1(partition_iterator, support_fraction):

    baskets_list = [ #used because partition_iterator is a single use data stream
        tuple(sorted(basket))
        for basket in partition_iterator
    ]

    if not baskets_list:
        return iter([])

    local_threshold = math.ceil(
        support_fraction * len(baskets_list)
    )

    counts = Counter()

    for basket in baskets_list:

        for item in basket:
            counts[(item,)] += 1

    freq = {
        itemset
        for itemset, count in counts.items()
        if count >= local_threshold
    }

    all_freq = set(freq)

    k = 2

    while freq:

        candidates = generate_candidates(
            freq,
            k
        )

        if not candidates:
            break

        candidate_set = set(candidates)

        counts = Counter()

        for basket in baskets_list:

            if len(basket) < k:
                continue

            for itemset in combinations(
                basket,
                k
            ):

                if itemset in candidate_set:
                    counts[itemset] += 1

        freq = {
            itemset
            for itemset, count in counts.items()
            if count >= local_threshold
        }

        all_freq.update(freq)

        k += 1

        if k > 4:
            break

    return iter(all_freq)

## 5) SON Algorithm

In [ ]:
candidates_rdd = (
    baskets
    .mapPartitions( #local_apriori_pass1 done on every partition
        lambda part:
            local_apriori_pass1(part, min_supp)
    )
    .distinct() #dups removal
    .persist(StorageLevel.MEMORY_AND_DISK) #save rdd in memory
)

print(
    "SON candidates:",
    candidates_rdd.count()
)

In [ ]:
n_candidates = candidates_rdd.count()

def basket_subsets(partition):
        for basket in partition:
            basket = tuple(sorted(set(basket)))
            for k in range(1, len(basket) + 1):
                for itemset in combinations(basket, k):
                    yield (itemset, 1)
basket_subset_counts = (
        baskets.mapPartitions(basket_subsets)
               .reduceByKey(lambda x, y: x + y)
)
candidates_keyed = candidates_rdd.map(lambda itemset: (itemset, None))

In [ ]:
candidate_counts_rdd = (
    basket_subset_counts
    .join(candidates_keyed)
    .map(lambda kv: (kv[0], kv[1][0]))
)

## 6) Filtering by frequent items

In [ ]:
freq_itemsets_rdd = (
    candidate_counts_rdd
    .filter(
        lambda x:
            x[1] >= glob_supp
    )
    .persist(
        StorageLevel.MEMORY_AND_DISK
    )
    
)

print(
    "Global freq itemsets:",
    freq_itemsets_rdd.count()
)

In [ ]:
for itemset, count in (
    freq_itemsets_rdd
    .takeOrdered(
        50,
        key=lambda x: (-x[1], x[0])
    )
):

    print(
        itemset,
        "->",
        count
    )

In [ ]:
pairs_and_triples = (
    freq_itemsets_rdd
    .filter(
        lambda x:
            len(x[0]) >= 2
    )
    .takeOrdered(
        50,
        key=lambda x:
            (-x[1], x[0])
    )
)

for itemset, count in pairs_and_triples:

    if len(itemset) == 2:
        itemset_type = "pair"
    elif len(itemset) == 3:
        itemset_type = "triple"
    else:
        itemset_type = f"group of {len(itemset)}"

    print(
        f"[{itemset_type}]",
        itemset, "->", count, "movies"
    )

## 7) Frequent itemsets

In [ ]:

print(f"Global freq itemsets ({min_supp*100}% supp): {freq_itemsets_rdd.count()}")
print(f"Found {len(pairs_and_triples)} freq pairs/triples")
for itemset, count in pairs_and_triples:
    if len(itemset) == 2: type = "pair"
    elif len(itemset) == 3: type = "triple"
    else: type = f"Group of {len(itemset)}"
    print(f"[{type}] {itemset} are togethere in {count} movies")


In [ ]:
for itemset, count in freq_itemsets_rdd.take(5):
    print(f"Actors: {itemset}")
    print(f"Total support: {count}")

## 8) Association rules

In [ ]:
#association rules A --> B
min_confidence = 0.75
def generate_rules(itemset_count):
    itemset, count = itemset_count
    if len(itemset) < 2:
        return

    for item in itemset:

        antecedent = tuple(
            x
            for x in itemset
            if x != item
        )

        consequent = (item,)

        yield (
            antecedent,
            consequent,
            count
        )


In [ ]:
rules_rdd = (
    freq_itemsets_rdd
    .flatMap(generate_rules)
)

In [ ]:
antecedent_support_rdd = (
    freq_itemsets_rdd
    .map(
        lambda x:
            (
                x[0],
                x[1]
            )
    )
)

In [ ]:
rules_with_confidence = (
    rules_rdd
    .map(
        lambda x:
            (
                x[0],
                (
                    x[1],
                    x[2]
                )
            )
    )
    .join(
        antecedent_support_rdd
    )
    .map(
        lambda x:
            (
                x[0], #antecedent
                x[1][0][0], #consequent
                x[1][0][1] / x[1][1] #confidence
            )
    )
    .filter(
        lambda x:
            x[2] >= min_confidence
    )
)

In [ ]:
consequent_support_rdd = (
    freq_itemsets_rdd
    .map(
        lambda x:
            (
                x[0],
                x[1]
            )
    )
)

In [ ]:
rules_with_interest = (
    rules_with_confidence
    .map(
        lambda x:
            (x[1],(x[0],x[2]) #moving cons first to use as a key
            )
    )
    .join(
        consequent_support_rdd
    )
    .map(
        lambda x:
            (
                x[1][0][0], #antecedent
                x[0], #consequent
                x[1][0][1], #confidence
                x[1][1] / n_baskets #support
            )
    )
    .map(
        lambda x:
            (
                x[0], #antecedent
                x[1], #consequent
                x[2], #confidence
                x[2] - x[3] #interest
            )
    )
)

In [ ]:
best_rules = (
    rules_with_interest
    .takeOrdered(
        20,
        key=lambda x:
            (-x[2], -x[3])
    )
)

for (
    antecedent,
    consequent,
    confidence,
    interest
) in best_rules:

    print(
        antecedent,
        "-->",
        consequent,
        "| confidence =",
        round(confidence, 3),
        "| interest =",
        round(interest, 3)
    )

## 9) Graph visualization

In [ ]:
G = nx.DiGraph()
for (
    antecedent,
    consequent,
    confidence,
    interest
) in best_rules:
    node_A = ", ".join(antecedent)
    node_B = ", ".join(consequent)
    G.add_edge(node_A, node_B, weight = confidence)

plt.figure(figsize=(12, 6))
pos = nx.spring_layout(G, k=0.5, seed = 1)
nx.draw_networkx_nodes(G, pos, node_size=2000, node_color='skyblue')
nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=15, edge_color='gray')
nx.draw_networkx_labels(G, pos, font_size=9, font_family='sans-serif')

plt.title("Graph of association rules between actors")
plt.axis('off')
plt.tight_layout()
plt.show()